### **Convolutional Neural Network (CNN)**
Corrected version of the course notebook (the original is kept as `CNN_original.ipynb`). Changes: seeds; a separate validation set (the test set is used only once at the end); LeNet-5 with the classic 2x2 pooling; the depthwise convolution now really uses `groups=in_channel`; AlexNet has its five convolutions and is used with 224x224 inputs; the training loop is a function; and every shape and parameter count is checked against the formulas of the chapter.

In [ ]:
import os, random, time
import numpy as np
import torch
import torch.nn as nn

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OFFLINE = os.environ.get("CNN_NOTEBOOK_OFFLINE") == "1"   # smoke test without downloading Fashion-MNIST

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def out_size(n, f, s=1, p=0):
    # (n - f + 2p)/s + 1, rounded down as PyTorch does
    return (n - f + 2 * p) // s + 1

#### Convolution layer: output size and parameters

In [ ]:
class SimpleConvLayer(nn.Module):
    def __init__(self, in_channel, output_channel, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channel, output_channel, kernel_size=kernel_size, stride=stride, padding=padding)

    def forward(self, x):
        return self.conv(x)

# the three examples of the original notebook, now with the formulas next to the measured values
for k, p, size in [(3, 1, 32), (5, 2, 32), (7, 3, 64)]:
    layer = SimpleConvLayer(3, 64, kernel_size=k, padding=p)
    y = layer(torch.randn(1, 3, size, size))
    expected = out_size(size, k, 1, p)
    n = count_parameters(layer)
    assert y.shape == (1, 64, expected, expected) and n == k * k * 3 * 64 + 64
    print(f"kernel {k}x{k}, padding {p}: output {tuple(y.shape)}, parameters {n} (= {k}*{k}*3*64 + 64)")

kernel 3x3, padding 1: output (1, 64, 32, 32), parameters 1792 (= 3*3*3*64 + 64)
kernel 5x5, padding 2: output (1, 64, 32, 32), parameters 4864 (= 5*5*3*64 + 64)
kernel 7x7, padding 3: output (1, 64, 64, 64), parameters 9472 (= 7*7*3*64 + 64)


In [ ]:
class DoubleConvLayer(nn.Module):
    # two parallel branches (3x3 and 7x7) that are added, followed by 2x2 max pooling
    def __init__(self, in_channel, output_channel):
        super().__init__()
        self.conv_3_3 = nn.Conv2d(in_channel, output_channel, kernel_size=3, stride=1, padding=1)
        self.conv_7_7 = nn.Conv2d(in_channel, output_channel, kernel_size=7, stride=1, padding=3)
        self.pooling = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

    def forward(self, x):
        return self.pooling(self.conv_3_3(x) + self.conv_7_7(x))

layer = DoubleConvLayer(3, 64)
y = layer(torch.randn(1, 3, 32, 32))
print("DoubleConvLayer:", tuple(y.shape), count_parameters(layer), "parameters (= 1792 + 9472; pooling has none)")

DoubleConvLayer: (1, 64, 16, 16) 11264 parameters (= 1792 + 9472; pooling has none)


#### Data: Fashion-MNIST with a validation split

In [ ]:
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

if OFFLINE:   # random images, only to test that the code runs; the accuracy is meaningless
    full_train = datasets.FakeData(size=2000, image_size=(1, 28, 28), num_classes=10, transform=ToTensor())
    test_data = datasets.FakeData(size=500, image_size=(1, 28, 28), num_classes=10, transform=ToTensor())
else:
    full_train = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor())
    test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

n_val = len(full_train) // 6                       # 10 000 of 60 000 images for validation
training_data, validation_data = random_split(full_train, [len(full_train) - n_val, n_val],
                                              generator=torch.Generator().manual_seed(SEED))
print(len(training_data), len(validation_data), len(test_data))
labels_map = {0: "T-Shirt", 1: "Trouser", 2: "Pullover", 3: "Dress", 4: "Coat",
              5: "Sandal", 6: "Shirt", 7: "Sneaker", 8: "Bag", 9: "Ankle Boot"}

In [ ]:
figure = plt.figure(figsize=(6, 6))
for i in range(1, 10):
    img, label = training_data[i]
    figure.add_subplot(3, 3, i)
    plt.title(labels_map[label]); plt.axis("off"); plt.imshow(img.squeeze(), cmap="gray")
plt.show()

In [ ]:
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_dataloader = DataLoader(validation_data, batch_size=256, shuffle=False)
test_dataloader = DataLoader(test_data, batch_size=256, shuffle=False)   # not shuffled, used once at the end

sample_input, sample_label = next(iter(train_dataloader))
print("input:", sample_input.shape, "labels:", sample_label.shape)    # (64, 1, 28, 28), (64,)

#### LeNet-5 (classic: 5x5 convolutions, 2x2 average pooling)

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, in_channel, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channel, 6, kernel_size=5)        # 28 -> 24
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)          # 24 -> 12
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)                # 12 -> 8
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)          # 8 -> 4
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

model_LeNet5 = LeNet5(in_channel=1, num_classes=10)
print(model_LeNet5)
print("parameters:", count_parameters(model_LeNet5))   # 44 426 (the original notebook had 42 890: 3x3 conv2, pool 5/2)
assert model_LeNet5(torch.zeros(1, 1, 28, 28)).shape == (1, 10)

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc1): Linear(in_features=256, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
  (relu): ReLU()
)
parameters: 44426


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss += nn.functional.cross_entropy(out, yb, reduction="sum").item()
        correct += (out.argmax(1) == yb).sum().item()
        n += len(yb)
    return loss / n, correct / n

def train_model(model, epochs, lr=0.01, momentum=0.9):
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        run_loss = correct = n = 0
        for xb, yb in train_dataloader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = nn.functional.cross_entropy(out, yb)
            loss.backward()
            opt.step()
            run_loss += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(yb)
        v_loss, v_acc = evaluate(model, val_dataloader)
        hist["train_loss"].append(run_loss / n); hist["train_acc"].append(correct / n)
        hist["val_loss"].append(v_loss); hist["val_acc"].append(v_acc)
        print(f"Epoch {epoch+1:2d} | train loss {run_loss/n:.4f} acc {correct/n:.4f} | val loss {v_loss:.4f} acc {v_acc:.4f}")
    print(f"Training time: {time.time() - t0:.1f} s")
    return hist

In [ ]:
EPOCHS = 1 if OFFLINE else 10
hist = train_model(model_LeNet5, EPOCHS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy loss"); ax[0].legend()
ax[1].plot(hist["train_acc"], label="train"); ax[1].plot(hist["val_acc"], label="validation")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# the test set is used once, after all choices are made
test_loss, test_acc = evaluate(model_LeNet5, test_dataloader)
print(f"Test accuracy of LeNet-5: {test_acc:.4f}")

In [ ]:
# predictions for nine test images (fixed indices, so the figure is reproducible)
model_LeNet5.to("cpu").eval()
figure = plt.figure(figsize=(6, 6))
for i in range(1, 10):
    img, label = test_data[i]
    with torch.no_grad():
        prediction = model_LeNet5(img.unsqueeze(0)).argmax(1).item()
    figure.add_subplot(3, 3, i)
    plt.title(f"Predicted: {labels_map[prediction]}\nTrue: {labels_map[label]}", fontsize=8)
    plt.axis("off"); plt.imshow(img.squeeze(), cmap="gray")
plt.show()

#### Standard, grouped and depthwise-separable convolution (see the chapter on lightweight CNNs)

In [ ]:
class SeparableConvLayer(nn.Module):
    # depthwise (groups = in_channel) 3x3 followed by pointwise 1x1
    def __init__(self, in_channel, output_channel, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channel, in_channel, kernel_size, stride, padding, groups=in_channel)
        self.pointwise = nn.Conv2d(in_channel, output_channel, kernel_size=1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

std_conv = nn.Conv2d(32, 64, 3, padding=1)
group_conv = nn.Conv2d(32, 64, 3, padding=1, groups=32)
sep_conv = SeparableConvLayer(32, 64)
x = torch.randn(1, 32, 28, 28)
for name, m in [("standard", std_conv), ("grouped (g=32)", group_conv), ("separable", sep_conv)]:
    print(f"{name:15s} output {tuple(m(x).shape)}  parameters {count_parameters(m)}")
assert (count_parameters(std_conv), count_parameters(group_conv), count_parameters(sep_conv)) == (18496, 640, 2432)

class SepLeNet5(LeNet5):
    def __init__(self, in_channel, num_classes):
        super().__init__(in_channel, num_classes)
        self.conv1 = SeparableConvLayer(in_channel, 6, kernel_size=5, padding=0)
        self.conv2 = SeparableConvLayer(6, 16, kernel_size=5, padding=0)

print("SepLeNet-5 parameters:", count_parameters(SepLeNet5(1, 10)), "vs LeNet-5:", count_parameters(LeNet5(1, 10)))

standard        output (1, 64, 28, 28)  parameters 18496


grouped (g=32)  output (1, 64, 28, 28)  parameters 640
separable       output (1, 64, 28, 28)  parameters 2432
SepLeNet-5 parameters: 42160 vs LeNet-5: 44426


#### Custom CNNs: standard and depthwise-separable blocks

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, in_channel, num_classes, block):
        super().__init__()
        self.features = nn.Sequential(
            block(in_channel, 32), nn.ReLU(), block(32, 64), nn.ReLU(), nn.MaxPool2d(2),
            block(64, 128), nn.ReLU(), block(128, 256), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(256 * 7 * 7, 50), nn.ReLU(), nn.Linear(50, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

std_block = lambda a, b: nn.Conv2d(a, b, 3, padding=1)
myStdCNN = CustomCNN(1, 10, std_block)
myDWSCNN = CustomCNN(1, 10, lambda a, b: SeparableConvLayer(a, b))
print("myStdCNN parameters: ", count_parameters(myStdCNN))     # 1 015 600
print("myDWSCNN parameters: ", count_parameters(myDWSCNN))     # 673 530 (the original code lacked groups=: 865 050)
print(myStdCNN(torch.zeros(1, 1, 28, 28)).shape, myDWSCNN(torch.zeros(1, 1, 28, 28)).shape)

myStdCNN parameters:  1015600
myDWSCNN parameters:  673530
torch.Size([1, 10]) torch.Size([1, 10])


#### AlexNet (five convolutions; designed for 224x224 inputs)

In [ ]:
class Alex_Net(nn.Module):
    def __init__(self, in_channel, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channel, 96, kernel_size=11, stride=4, padding=2), nn.ReLU(), nn.MaxPool2d(3, 2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(3, 2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(),     # conv4 (was overwritten in the original)
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),     # conv5
            nn.MaxPool2d(3, 2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(), nn.Linear(4096, 4096), nn.ReLU(),
            nn.Linear(4096, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

model_AlexNet = Alex_Net(in_channel=1, num_classes=10)
print("AlexNet parameters:", count_parameters(model_AlexNet))   # 58 299 082; 93.6 % are in the dense layers
print("224x224 input ->", tuple(model_AlexNet(torch.zeros(1, 1, 224, 224)).shape))
try:
    model_AlexNet(torch.zeros(1, 1, 28, 28))                    # the original notebook fed 28x28 images
except RuntimeError as e:
    print("28x28 input fails:", str(e).split(".")[0])

AlexNet parameters: 58299082
224x224 input -> (1, 10)
28x28 input fails: Given input size: (256x2x2)


To train AlexNet on Fashion-MNIST the images must be resized to 224x224 (`transforms.Resize(224)`), which is slow on a CPU. The cell below trains on a small subset for one epoch, only to show the procedure; it is not part of the chapter's results.

In [ ]:
from torch.utils.data import Subset
from torchvision import transforms
if not OFFLINE:
    tf = transforms.Compose([transforms.Resize(224), transforms.ToTensor()])
    big = datasets.FashionMNIST(root="data", train=True, download=True, transform=tf)
    subset = Subset(big, range(2000))
    train_dataloader = DataLoader(subset, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(SEED))
    val_dataloader = DataLoader(Subset(big, range(2000, 2500)), batch_size=64)
    hist_alex = train_model(model_AlexNet, epochs=1, lr=0.01)